# NeuroSkin AI — RAG Pipeline v3

**Architecture:**
```
ONE-TIME SETUP (run once, then stored forever in MongoDB Atlas):
DermNet + MedlinePlus + WikiDoc + AAD + NHS + PCDS
           ↓
   Parallel WebLoader + RecursiveCharacterTextSplitter
           ↓
   Nomic Embeddings (nomic-embed-text-v1.5)  ← API-based, works on deployment
           ↓
   MongoDB Atlas Vector Store  ← cloud, persists forever, no rebuild needed

─────────────────────────────────────────────────────────────────

EVERY REQUEST:
User symptoms + CNN predictions
           ↓
   Query → Nomic Embeddings
           ↓
   MongoDB Atlas Vector Search  (fetches top 10 candidates)
           ↓
   Cohere Reranker  (reranks → keeps top 3 most relevant)
           ↓
   Groq LLM (llama-3.1-8b-instant)  ← context + symptoms
           ↓
   JSON result → React frontend
```

**Changes from v2:**
- `ChromaDB` → `MongoDB Atlas` (cloud, no rebuild on deployment)
- `SentenceTransformers` → `Nomic Embeddings API` (no local model, works deployed)
- Added `Cohere Rerank` as a second-pass filter after vector search
- Fetch 10 candidates from MongoDB, rerank to top 3 → cleaner LLM context
- Tighter, safer prompt template with hallucination guards
- Extra test cases and batch runner kept, all commented out

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

packages = [
    # LangChain core stack
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-groq",
    "langchain-text-splitters",
    "langchain-mongodb",       # MongoDB Atlas vector store integration
    "langchain-nomic",         # Nomic embeddings integration
    "langchain-cohere",        # Cohere reranker integration
    # Database
    "pymongo",                 # MongoDB driver
    # Reranker
    "cohere",
    # Web scraping
    "beautifulsoup4",
    "requests",
    "lxml",
    # Utilities
    "python-dotenv",
    "nest-asyncio",
]

for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--upgrade"],
        check=False
    )

# print("All packages installed.")
# print("IMPORTANT: If first install, restart kernel then run all cells.")

## Cell 2 — Environment & API Keys

**You need to fill in 4 API keys here:**

| Key | Where to get it |
|---|---|
| `GROQ_API_KEY` | https://console.groq.com → API Keys |
| `MONGODB_URI` | https://cloud.mongodb.com → Connect → Drivers |
| `NOMIC_API_KEY` | https://atlas.nomic.ai → Settings → API Keys |
| `COHERE_API_KEY` | https://dashboard.cohere.com → API Keys |

In [ ]:
import os
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()
load_dotenv()  # .env file se keys load hogi

# ── API Keys — .env file mein set karo (kabhi hardcode mat karo!) ─────────────
# .env file banao aur yeh lines add karo:
#   GROQ_API_KEY=your_key_here        # console.groq.com
#   MONGODB_URI=your_uri_here         # cloud.mongodb.com
#   NOMIC_API_KEY=your_key_here       # atlas.nomic.ai
#   COHERE_API_KEY=your_key_here      # dashboard.cohere.com

GROQ_API_KEY   = os.getenv("GROQ_API_KEY")
MONGODB_URI    = os.getenv("MONGODB_URI")
NOMIC_API_KEY  = os.getenv("NOMIC_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

assert GROQ_API_KEY,   "GROQ_API_KEY not set — .env file mein add karo"
assert MONGODB_URI,    "MONGODB_URI not set — .env file mein add karo"
assert NOMIC_API_KEY,  "NOMIC_API_KEY not set — .env file mein add karo"
assert COHERE_API_KEY, "COHERE_API_KEY not set — .env file mein add karo"

# MongoDB collection settings
MONGO_DB_NAME         = "neuroskin"
MONGO_COLLECTION_NAME = "dermatology_v3"
MONGO_INDEX_NAME      = "neuroskin_vector_index"

print("✅ Environment ready.")


## Cell 3 — Nomic Embeddings

Nomic `nomic-embed-text-v1.5` is API-based — no local model download, no GPU needed,
works on any deployment platform (Railway, Render, Vercel, etc.).

768-dimensional embeddings with Matryoshka training (can use fewer dims if needed).

In [3]:
from langchain_nomic import NomicEmbeddings

embeddings = NomicEmbeddings(
    model="nomic-embed-text-v1.5",
    nomic_api_key=NOMIC_API_KEY,
    inference_mode="remote",   # API-based — works everywhere including deployment
    dimensionality=768,        # Full 768-dim for best accuracy
)

# _test = embeddings.embed_query("eczema itching dry skin")
# print(f"Nomic embeddings ready. Dimension: {len(_test)}")

## Cell 4 — Document Loaders

Sources: DermNet + MedlinePlus + WikiDoc + AAD + NHS + PCDS (parallel loading).
Cached to a local pickle so we only re-fetch if the cache file is deleted.

In [18]:
import pickle, os, time, concurrent.futures
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

# ── URL Lists ──────────────────────────────────────────────────────────────────
DERMNET_URLS = [
    "https://dermnetnz.org/topics/melanoma",
    "https://dermnetnz.org/topics/basal-cell-carcinoma",
    "https://dermnetnz.org/topics/actinic-keratosis",
    "https://dermnetnz.org/topics/seborrhoeic-keratosis",
    "https://dermnetnz.org/topics/acne",
    "https://dermnetnz.org/topics/atopic-dermatitis",
    "https://dermnetnz.org/topics/psoriasis",
    "https://dermnetnz.org/topics/tinea-corporis",
    "https://dermnetnz.org/topics/differential-diagnosis-of-eczema",
]

MEDLINEPLUS_URLS = [
    "https://medlineplus.gov/fungalinfections.html",
    "https://medlineplus.gov/melanoma.html",
    "https://medlineplus.gov/skincancer.html",
    "https://medlineplus.gov/acne.html",
    "https://medlineplus.gov/eczema.html",
    "https://medlineplus.gov/psoriasis.html",
    "https://medlineplus.gov/rashes.html",
]

AAD_URLS = [
    "https://www.aad.org/public/diseases/skin-cancer/types/common/melanoma",
    "https://www.aad.org/public/diseases/acne",
    "https://www.aad.org/public/diseases/eczema/types/atopic-dermatitis",
    "https://www.aad.org/public/diseases/psoriasis",
    "https://www.aad.org/public/diseases/actinic-keratosis",
]

NHS_URLS = [
    "https://www.nhs.uk/conditions/basal-cell-skin-cancer/",
    "https://www.nhs.uk/conditions/melanoma-skin-cancer/",
    "https://www.nhs.uk/conditions/acne/",
    "https://www.nhs.uk/conditions/atopic-eczema/",
    "https://www.nhs.uk/conditions/psoriasis/",
    "https://www.nhs.uk/conditions/ringworm/",
]

PCDS_URLS = [
    "https://www.pcds.org.uk/clinical-guidance/actinic-keratosis",
]

DOCS_CACHE = "./neuroskin_docs_cache.pkl"


def _fetch_one(url: str, source_tag: str) -> Document:
    """Fetch a single URL with BeautifulSoup, return a Document or None on failure."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    try:
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        if len(text) < 200:
            return None
        return Document(
            page_content=text,
            metadata={"source": url, "source_tag": source_tag},
        )
    except Exception as exc:
        print(f"  WARN: {url} → {exc}")
        return None


def load_urls_parallel(urls: list, source_tag: str, max_workers: int = 6) -> list:
    """Parallel-fetch a list of URLs, return list of Documents."""
    docs = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(_fetch_one, url, source_tag): url for url in urls}
        for future in concurrent.futures.as_completed(futures):
            doc = future.result()
            if doc:
                docs.append(doc)
    print(f"  [{source_tag}] fetched {len(docs)}/{len(urls)}")
    return docs


# ── Load / Cache ───────────────────────────────────────────────────────────────
if os.path.exists(DOCS_CACHE):
    with open(DOCS_CACHE, "rb") as f:
        all_web_docs = pickle.load(f)
    print(f"Docs loaded from cache: {len(all_web_docs)}")
else:
    t0 = time.time()
    dermnet_docs     = load_urls_parallel(DERMNET_URLS,     "dermnet")
    medlineplus_docs = load_urls_parallel(MEDLINEPLUS_URLS, "medlineplus")
    aad_docs         = load_urls_parallel(AAD_URLS,         "aad")
    nhs_docs         = load_urls_parallel(NHS_URLS,         "nhs")
    pcds_docs        = load_urls_parallel(PCDS_URLS,        "pcds")

    all_web_docs = (
        dermnet_docs + medlineplus_docs + wikidoc_docs
        + aad_docs + nhs_docs + pcds_docs
    )

    with open(DOCS_CACHE, "wb") as f:
        pickle.dump(all_web_docs, f)
    # print(f"Docs fetched and cached: {len(all_web_docs)} in {time.time()-t0:.1f}s")
    # print("TIP: Delete neuroskin_docs_cache.pkl to force a re-fetch.")

Docs loaded from cache: 28


## Cell 5 — Text Splitter

In [19]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter


def clean_doc_text(doc: Document) -> Document:
    """Remove excess whitespace and cookie/privacy noise from scraped text."""
    text = doc.page_content
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\t+", " ", text)
    for pattern in [r"Skip to.*?\n", r"Cookie.*?\n", r"Privacy.*?\n"]:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    doc.page_content = text.strip()
    return doc


cleaned_docs = [clean_doc_text(doc) for doc in all_web_docs]
cleaned_docs = [doc for doc in cleaned_docs if len(doc.page_content) > 200]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    length_function=len,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
)

chunks = splitter.split_documents(cleaned_docs)

# print(f"Input docs: {len(cleaned_docs)} | Output chunks: {len(chunks)}")

## Cell 6 — MongoDB Atlas Vector Store

**One-time setup:** The first time this cell runs it builds the vector store in MongoDB Atlas.
Every time after that it just connects to the existing collection — no rebuild needed.

**Before running:** Create a free MongoDB Atlas cluster at https://cloud.mongodb.com, then:
1. Go to Atlas Search → Create Search Index
2. Choose "Vector Search", database = `neuroskin`, collection = `dermatology_v3`
3. Index name = `neuroskin_vector_index`, dimensions = `768`, similarity = `cosine`

In [20]:
import time
from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch

# Connect to MongoDB Atlas
mongo_client = MongoClient(MONGODB_URI)
collection   = mongo_client[MONGO_DB_NAME][MONGO_COLLECTION_NAME]

# Check if the collection already has data
existing_count = collection.count_documents({})

if existing_count > 0:
    # Already populated — just connect, no re-embedding needed
    vectorstore = MongoDBAtlasVectorSearch.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection=collection,
    index_name=MONGO_INDEX_NAME,
    text_key="text",           # ← add this
    embedding_key="embedding", # ← add this
)
    # print(f"MongoDB connected. Existing vectors: {existing_count}")

else:
    # First-time setup — embed all chunks and store in MongoDB Atlas
    print(f"First-time setup: embedding {len(chunks)} chunks into MongoDB Atlas...")
    t0 = time.time()

    vectorstore = MongoDBAtlasVectorSearch.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection=collection,
        index_name=MONGO_INDEX_NAME,
    )

    # print(f"MongoDB Atlas vector store built in {time.time()-t0:.1f}s")
    # print(f"Vectors stored: {collection.count_documents({})}")
    # print("NOTE: This was a one-time setup. Future runs will skip this step.")

C:\Users\kunjp\AppData\Roaming\Python\Python313\site-packages\pymongo\pyopenssl_context.py:348: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  _crypto.X509.from_cryptography(x509.load_der_x509_certificate(cert))


## Cell 7 — Retriever with Cohere Reranker

Two-stage retrieval:
1. **MongoDB Vector Search** fetches top 10 candidates by embedding similarity
2. **Cohere Reranker** re-scores them against the exact query, keeps top 3

Result: the 3 docs sent to the LLM are far more relevant than raw vector search alone.

In [31]:
import cohere
from functools import lru_cache
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import List

# Stage 1: MongoDB vector search — fetch 10 candidates per query
mongo_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 15},  # broader net, reranker will filter to best 3
)

# Stage 2: Cohere reranker client
cohere_client = cohere.Client(api_key=COHERE_API_KEY)

TOP_N_AFTER_RERANK = 5  # how many docs to pass to the LLM


def rerank_documents(query: str, docs: List[Document]) -> List[Document]:
    """
    Use Cohere Rerank to re-score and sort documents by relevance to the query.
    Returns the top TOP_N_AFTER_RERANK most relevant documents.
    """
    if not docs:
        return []

    doc_texts = [doc.page_content for doc in docs]

    response = cohere_client.rerank(
        model="rerank-english-v3.0",  # latest Cohere reranker
        query=query,
        documents=doc_texts,
        top_n=TOP_N_AFTER_RERANK,
    )

    # Return docs sorted by rerank score (highest relevance first)
    reranked = [docs[result.index] for result in response.results]
    return reranked


@lru_cache(maxsize=64)
def _cached_retrieve_and_rerank(query: str) -> tuple:
    """Cache results so repeated identical queries don't hit the APIs again."""
    candidates = mongo_retriever.invoke(query)   # step 1: broad vector search
    top_docs   = rerank_documents(query, candidates)  # step 2: rerank to top 3
    return tuple(top_docs)


class RerankRetriever(BaseRetriever):
    """LangChain-compatible retriever: MongoDB vector search + Cohere rerank."""

    def _get_relevant_documents(self, query: str) -> List[Document]:
        return list(_cached_retrieve_and_rerank(query))


retriever = RerankRetriever()

# _test = retriever.invoke("eczema symptoms itching")
# print(f"Retriever ready. Test returned {len(_test)} docs after rerank.")

## Cell 8 — LLM Setup (Groq)

In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,       # deterministic — critical for medical output
    max_tokens=1500,
    api_key=GROQ_API_KEY,
)

# _t = llm.invoke("Say READY")
# print(f"LLM ready: {_t.content.strip()}")

## Cell 9 — Prompt Template

Designed to:
- Ground every answer strictly in retrieved medical context (no hallucination)
- Never override the CNN prediction — only revise probability based on symptom match
- Stay within the 3-condition scope given by the CNN
- Fail gracefully by including a mandatory disclaimer and doctor referral logic
- Produce consistent, parseable JSON every time

In [33]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

SYSTEM_PROMPT = """You are NeuroSkin AI, a dermatology screening assistant.
A CNN image model has already analyzed a patient skin photo and returned its top-3 predictions.
Your job is to refine those predictions using the retrieved medical context and the patient's symptoms.

STRICT RULES — follow every one of these without exception:

1. SOURCE DISCIPLINE: "Prefer information from the retrieved context. If the context does not cover a field, use general dermatology knowledge but keep it conservative."

2. PREDICTION SCOPE: You may ONLY revise the probabilities of the 3 conditions the CNN gave you.
   Do NOT introduce new conditions. Do NOT rename CNN conditions.

3. PROBABILITY RULE: The three revised_confidence values in top_3_predictions MUST sum to exactly 100.
   Shift probability TOWARD the condition whose symptoms best match the patient's answers.
   Never change the ORDER of the top-3 conditions — only the probability values.

4. SEVERITY RULE: Use the CNN severity as a baseline. You may escalate severity (never downgrade)
   only if the patient reports symptoms like: rapid spread, bleeding, open wounds, fever,
   or signs matching urgent conditions (e.g., melanoma ABCDE criteria).

5. NO DIAGNOSIS: You are a screening assistant. Never state a definitive diagnosis.
   Always use language like "most consistent with", "likely", "suggests".

6. SAFE FALLBACK: If the medical context is too vague to support a specific claim,
   output the safest conservative response and set see_doctor to true.

7. FORMAT: Return ONLY valid JSON matching the schema below. No markdown. No explanation.
   No text before or after the JSON.

Output JSON schema:
{{
  "primary_condition": "<most likely condition from CNN top-3>",
  "summary": "<2-sentence plain-English screening summary — use 'suggests' not 'is'>",
  "top_3_predictions": [
    {{"condition": "<name>", "cnn_confidence": <float>, "revised_confidence": <float>, "symptom_match": "high|medium|low"}}
  ],
  "severity": "mild|moderate|severe|urgent",
  "severity_reason": "<one sentence grounded in the context or reported symptoms>",
  "symptoms_present": ["<symptom found in patient answers>"],
  "triggers": ["<3-5 evidence-based triggers from context>"],
  "contagion_risk": "yes — <one-line reason>  OR  no — <one-line reason>",
  "home_care_steps": ["<3-4 concise evidence-based steps from context>"],
  "things_to_avoid": ["<avoid 1 from context>"],
  "doctor_talking_points": ["<point 1>", "<point 2>", "<point 3>"],
  "see_doctor": true|false,
  "see_doctor_if": "<specific warning sign that means go now>",
  "urgency_days": <integer — 0 if urgent, else days before doctor visit recommended>,
  "typical_recovery": "<brief timeline from context>",
  "disclaimer": "AI-assisted screening only. Not a medical diagnosis. Consult a qualified dermatologist."
}}"""

combined_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template(
        "CNN Top-3 Predictions: {top_3_json}\n"
        "CNN Severity Estimate: {severity}\n"
        "Patient Symptoms: {symptom_answers}\n\n"
        "Retrieved Medical Context:\n{context}\n\n"
        "Generate the full screening report JSON:"
    ),
])

# print("Prompt ready.")

## Cell 10 — Chains

In [34]:
import json, re
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


def safe_json_parse(text) -> dict:
    """Strip markdown fences if any and parse JSON. Falls back to raw on failure."""
    if isinstance(text, dict):
        return text
    text = re.sub(r"```json\s*", "", str(text))
    text = re.sub(r"```\s*", "", text)
    try:
        return json.loads(text.strip())
    except Exception:
        return {"raw": text.strip()}


# Single chain: prompt → LLM → string → JSON
pipeline_chain = (
    combined_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(safe_json_parse)
)

# print("Chain ready.")

## Cell 11 — Pipeline Function

Main entry point. Call `neuroskin_full_pipeline()` with the CNN outputs.

In [35]:
import json

# If the gap between CNN top-1 and top-2 confidence is <= this, the CNN is uncertain
CNN_CONFUSION_THRESHOLD = 15.0


def format_retrieved_docs(docs) -> str:
    """Format retrieved docs into a clean numbered context string for the LLM."""
    sections = []
    for i, doc in enumerate(docs):
        src = doc.metadata.get("source_tag", "unknown")
        sections.append(f"[Source {i+1} — {src}]\n{doc.page_content.strip()}")
    return "\n\n".join(sections)


def is_cnn_confused(top_3_predictions: list) -> bool:
    """Return True if the CNN top-2 confidence gap is too small to trust."""
    if len(top_3_predictions) < 2:
        return False
    gap = top_3_predictions[0]["confidence"] - top_3_predictions[1]["confidence"]
    return gap <= CNN_CONFUSION_THRESHOLD


def neuroskin_full_pipeline(
    top_3_predictions: list,
    severity: str,
    symptom_answers: str
) -> dict:
    """
    Main pipeline function.

    Args:
        top_3_predictions : [{"condition": "eczema", "confidence": 72.4}, ...]
        severity          : CNN severity estimate string (e.g. "moderate")
        symptom_answers   : Patient symptom string from frontend questionnaire

    Returns:
        dict with revised top-3 probabilities + full clinical screening report

    If CNN is confused (top-2 gap <= 20%), returns early with see_doctor=True
    without calling the LLM.
    """

    # ── CNN Confusion Gate ─────────────────────────────────────────────────────
    # If the model can't distinguish top-2, don't run the full pipeline.
    # Return a safe early result telling the user to see a doctor.
    if is_cnn_confused(top_3_predictions):
        top1 = top_3_predictions[0]["condition"]
        top2 = top_3_predictions[1]["condition"]
        gap  = top_3_predictions[0]["confidence"] - top_3_predictions[1]["confidence"]
        return {
            "primary_condition": "inconclusive",
            "summary": (
                f"The AI image model could not confidently distinguish between "
                f"{top1} and {top2} (confidence gap: {gap:.1f}%). "
                "A dermatologist examination is required for accurate diagnosis."
            ),
            "top_3_predictions": [
                {
                    "condition": p["condition"],
                    "cnn_confidence": p["confidence"],
                    "revised_confidence": round(p["confidence"], 1),
                    "symptom_match": "unknown",
                }
                for p in top_3_predictions
            ],
            "cnn_confused": True,
            "severity": severity,
            "severity_reason": "CNN model was unable to distinguish between top conditions.",
            "symptoms_present": [],
            "triggers": ["Unknown — diagnosis required first"],
            "contagion_risk": "Unknown — diagnosis required first",
            "home_care_steps": [
                "Do not self-diagnose or self-medicate.",
                "Photograph the affected area in good lighting.",
                "Book an appointment with a dermatologist as soon as possible.",
            ],
            "things_to_avoid": ["Avoid applying unknown creams until diagnosed."],
            "doctor_talking_points": [
                "Show the AI results and explain that the model was uncertain.",
                "Describe how long the condition has been present and any changes.",
                "List all symptoms including itching, pain, or spreading.",
            ],
            "see_doctor": True,
            "see_doctor_if": "Condition is already inconclusive — please see a doctor now.",
            "urgency_days": 3,
            "typical_recovery": "Depends on confirmed diagnosis.",
            "disclaimer": "AI-assisted screening only. Not a medical diagnosis. Consult a qualified dermatologist.",
        }

    # ── Normal Pipeline ────────────────────────────────────────────────────────

    # Build a focused query: condition names + severity + patient symptoms
    conditions_str = " ".join([p["condition"] for p in top_3_predictions])
    query = f"{conditions_str} skin symptoms {severity} {symptom_answers}"

    # Retrieve + rerank: MongoDB (top 10) → Cohere rerank (top 3)
    docs    = retriever.invoke(query)
    context = format_retrieved_docs(docs)

    # Run the LLM chain
    result = pipeline_chain.invoke({
        "top_3_json":      json.dumps(top_3_predictions),
        "severity":        severity,
        "symptom_answers": symptom_answers,
        "context":         context,
    })

    result["cnn_confused"] = False
    return result


# print("Pipeline ready: neuroskin_full_pipeline(top_3_predictions, severity, symptom_answers)")
# print(f"CNN confusion threshold: gap <= {CNN_CONFUSION_THRESHOLD}% → early return with see_doctor=True")

## Cell 12 — Test Run (Single)

In [36]:
# import json

# TEST_INPUT = {
#     "top_3_predictions": [
#         {"condition": "eczema",    "confidence": 72.4},
#         {"condition": "psoriasis", "confidence": 18.3},
#         {"condition": "ringworm",  "confidence": 9.3},
#     ],
#     "severity": "moderate",
#     "symptom_answers": (
#         "Auspitz sign (pinpoint bleeding when scales are scraped), "
#         "chronic symmetric pattern like raindrops."
#     ),
# }

# result = neuroskin_full_pipeline(**TEST_INPUT)
# print(json.dumps(result, indent=2))

## Cell 13 — FastAPI Integration Helper

In [37]:
FASTAPI_SNIPPET = '''
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
from neuroskin_rag import neuroskin_full_pipeline

app = FastAPI()

class CNNPrediction(BaseModel):
    condition: str
    confidence: float

class PredictionInput(BaseModel):
    top_3_predictions: List[CNNPrediction]
    severity: str
    symptom_answers: str

@app.post("/analyze")
async def analyze(data: PredictionInput):
    result = neuroskin_full_pipeline(
        top_3_predictions = [p.dict() for p in data.top_3_predictions],
        severity          = data.severity,
        symptom_answers   = data.symptom_answers,
    )
    return result
'''

# print(FASTAPI_SNIPPET)
# print("To export as module:")
# print("  jupyter nbconvert --to script neuroskin_rag_pipeline_v3.ipynb --output neuroskin_rag")

## Cell 14 — Batch Test

In [38]:
import json, time

BATCH_TEST_CASES = [
    {
        "top_3_predictions": [
            {"condition": "melanoma",           "confidence": 81.2},
            {"condition": "basal cell carcinoma", "confidence": 12.1},
            {"condition": "actinic keratosis",  "confidence": 6.7},
        ],
        "severity": "severe",
        "symptom_answers": "Dark irregular mole growing fast. Asymmetric. Multiple colors."
    },
    {
        "top_3_predictions": [
            {"condition": "acne",     "confidence": 92.0},
            {"condition": "eczema",   "confidence": 5.0},
            {"condition": "psoriasis","confidence": 3.0},
        ],
        "severity": "moderate",
        "symptom_answers": "Painful cysts on chin and forehead. Oily skin. Teenage patient."
    },
    {
        "top_3_predictions": [
            {"condition": "eczema",    "confidence": 62.4},
            {"condition": "psoriasis", "confidence": 28.3},
            {"condition": "ringworm",  "confidence": 9.3},
        ],
        "severity": "moderate",
        "symptom_answers": "Auspitz sign (pinpoint bleeding when scales are scraped), chronic symmetric raindrop pattern."
    },
    # ── CNN confusion test — gap between top-1 and top-2 is only 8% ────────────
    {
        "top_3_predictions": [
            {"condition": "eczema",    "confidence": 44.0},
            {"condition": "psoriasis", "confidence": 36.0},
            {"condition": "ringworm",  "confidence": 20.0},
        ],
        "severity": "moderate",
        "symptom_answers": "Itchy red patches on arms. Unclear onset."
    },
]

batch_results = {}

for i, test in enumerate(BATCH_TEST_CASES):
    primary = test["top_3_predictions"][0]["condition"]
    gap = test["top_3_predictions"][0]["confidence"] - test["top_3_predictions"][1]["confidence"]
    print(f"[{i+1}/{len(BATCH_TEST_CASES)}] {primary}... (top-2 gap: {gap:.1f}%)")

    for attempt in range(3):
        try:
            result = neuroskin_full_pipeline(**test)
            batch_results[primary] = {
                "status": "OK",
                "confused": result.get("cnn_confused", False),
                "primary": result.get("primary_condition"),
                "severity": result.get("severity"),
                "top_3": [
                    (p["condition"], p.get("revised_confidence", p.get("cnn_confidence", 0.0)))
                    for p in result.get("top_3_predictions", [])
                ],
            }
            break
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(20 * (attempt + 1))
            else:
                batch_results[primary] = {"status": f"ERROR: {str(e)[:60]}"}
                break

    time.sleep(10)

print("\nBATCH RESULTS")
print("-" * 80)
for condition, res in batch_results.items():
    if res["status"] == "OK":
        if res.get("confused"):
            print(f"  CNN CONFUSED  {condition:25s} → see_doctor=True  (inconclusive)")
        else:
            top3_str = " | ".join([f"{c}:{p:.1f}%" for c, p in res.get("top_3", [])])
            print(f"  OK  {condition:25s} severity:{str(res.get('severity') or 'unknown'):10s} top3:[{top3_str}]")
    else:
        print(f"  ERR {condition:25s} {res['status']}")

[1/4] melanoma... (top-2 gap: 69.1%)
[2/4] acne... (top-2 gap: 87.0%)
[3/4] eczema... (top-2 gap: 34.1%)
[4/4] eczema... (top-2 gap: 8.0%)

BATCH RESULTS
--------------------------------------------------------------------------------
  OK  melanoma                  severity:urgent     top3:[melanoma:85.0% | basal cell carcinoma:8.0% | actinic keratosis:7.0%]
  OK  acne                      severity:moderate   top3:[acne:80.0% | eczema:10.0% | psoriasis:10.0%]
  CNN CONFUSED  eczema                    → see_doctor=True  (inconclusive)


In [29]:
import json
print(json.dumps(batch_results, indent=2))

{
  "melanoma": {
    "status": "OK",
    "confused": false,
    "primary": "melanoma",
    "severity": "urgent",
    "top_3": [
      [
        "melanoma",
        95.0
      ],
      [
        "basal cell carcinoma",
        4.0
      ],
      [
        "actinic keratosis",
        1.0
      ]
    ]
  },
  "acne": {
    "status": "OK",
    "confused": false,
    "primary": "acne",
    "severity": "moderate",
    "top_3": [
      [
        "acne",
        80.0
      ],
      [
        "eczema",
        10.0
      ],
      [
        "psoriasis",
        10.0
      ]
    ]
  },
  "eczema": {
    "status": "OK",
    "confused": true,
    "primary": "inconclusive",
    "severity": "moderate",
    "top_3": [
      [
        "eczema",
        44.0
      ],
      [
        "psoriasis",
        36.0
      ],
      [
        "ringworm",
        20.0
      ]
    ]
  }
}


In [30]:
result = neuroskin_full_pipeline(
    top_3_predictions=[
        {"condition": "melanoma", "confidence": 81.2},
        {"condition": "basal cell carcinoma", "confidence": 12.1},
        {"condition": "actinic keratosis", "confidence": 6.7},
    ],
    severity="severe",
    symptom_answers="Dark irregular mole growing fast. Asymmetric. Multiple colors."
)
print(json.dumps(result, indent=2))

{
  "primary_condition": "melanoma",
  "summary": "This dark irregular mole growing fast and asymmetrically suggests melanoma, a serious skin cancer. It's essential to consult a dermatologist for a proper evaluation and diagnosis.",
  "top_3_predictions": [
    {
      "condition": "melanoma",
      "cnn_confidence": 81.2,
      "revised_confidence": 95.0,
      "symptom_match": "high"
    },
    {
      "condition": "basal cell carcinoma",
      "cnn_confidence": 12.1,
      "revised_confidence": 4.0,
      "symptom_match": "low"
    },
    {
      "condition": "actinic keratosis",
      "cnn_confidence": 6.7,
      "revised_confidence": 1.0,
      "symptom_match": "low"
    }
  ],
  "severity": "urgent",
  "severity_reason": "The patient reports a dark irregular mole growing fast and asymmetrically, which is a concerning sign for melanoma.",
  "symptoms_present": [
    "Dark irregular mole growing fast",
    "Asymmetric",
    "Multiple colors"
  ],
  "triggers": [
    "Rapid growth",